In [13]:
#Baseline Algorithms in Flat Environment
from Multi_Agent_Environment_Flat import CustomEnvironmentFlat
from pettingzoo.test import parallel_api_test
import gymnasium as gym
env = CustomEnvironmentFlat()
parallel_api_test(env, num_cycles=1_000_000)


Passed Parallel API test


In [14]:
#Random Policy
import random
import numpy as np
# Initialize the env
episode_rewards=[]
# Use a fixed agent list
def RandomPolicy(env,seed):
    all_agents = ["satellite1", "satellite2", "satellite3"]
    agent_rewards = {agent: [] for agent in all_agents}
    # Track done flags
    obs, info = env.reset(seed)
    
    print(obs)
    print(info)
    terminated = {agent: False for agent in all_agents}
    truncated = {agent: False for agent in all_agents}
    total_rewards = {agent: 0 for agent in all_agents}
    while not all([terminated[a] or truncated[a] for a in all_agents]):
        actions = {
            agent: env.action_space(agent).sample()
            for agent in all_agents
            if not (terminated[agent] or truncated[agent])
        }
        #print("Agent Actions")
        #print(actions)
        observations, rewards, terminated, truncated, info = env.step(actions)
        for agent, r in rewards.items():
            total_rewards[agent] += r
        
        print("Rewards")
        print(rewards)
  
    for agent in all_agents:
        agent_rewards[agent].append(total_rewards.get(agent, 0))
    episode_rewards.append(np.mean(list(total_rewards.values())))
    Overall_Delivered_Packets=info['Delivered_Packets']
    Overall_Excess_Energy=info['Total_Energy_Expended']
    Overall_Number_of_Contacts_Used=info['Total_Number_of_Contacts_Used']
    agent_specific_delivered_packets=info['satellite_delivered_packets']
    agent_specific_excess_energy=info['satellite_energy_expended']
    agent_specific_num_of_contacts=info['satellite_number_of_contacts_used']
    num_of_collisions=info['Num_of_Collisions']
    env.close()
    return Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts
        

In [15]:
#Adaptive Sorting Heuristic
def AdaptiveSorting(env,seed):
    init_observations,init_info=env.reset()
    #print("Observation info")
    #print(init_observations)
    #Next need to conduct necessary calculations to obtain the strategy for the actions
    #With full observability we can just take satellite 1's observation space as we can see everything
    #that satellite 2 and 3 can see
    sat1_init_observations=init_observations["satellite1"]
    contact_plan=sat1_init_observations["central_observation_matrix"]
    #Need to add order identifier to each contact
    contact_count=0
    for contact in contact_plan:
        contact.append(contact_count)
        contact_count+=1
    #Need to manage all the satellite buffers and minimize collisions
    #Dictionary that contains the remaining packets to send
    satellite_buffers=sat1_init_observations["remaining_satellite_data"]
    print(satellite_buffers)
    #Next need to convert this to something easily handled in the heuristic algorithm
    all_agents = ["satellite1", "satellite2", "satellite3"]
    satellite_buffers_algo=[]
    for a in all_agents:
        satellite_buffers_algo.append(satellite_buffers[a])
    
    
    
    #Next we need to sort the contact plan by cloud cover value
    sorted_contact_plan=sorted(contact_plan,key=lambda row: row[0])
    print("Sorted Matrix")
    print(sorted_contact_plan)
    #Next we need to allocate the contacts based upon the buffers of the satellites
    estimated_capacity=0
    for i in sorted_contact_plan:
        if i[0]>= 0:
            estimated_capacity=round(i[0]*10)
    
    print("Estimated Capacity")
    print(estimated_capacity)
    
    #Now we have our initial action strategy
    #We implement while loop here
    actions=[]#Options are 0,1,2,3
    
    for contact in sorted_contact_plan:
        contact_action_options=[]
        if i[0]>=0:
            for l in range(0,2):
                contact_action_options.append(i[2+l])
            #Next we need to check how much data is left on each satellite that can transmit
            maximum_data_remaining=0
            satellite_counter=0
            max_satellite_index=None
            for sat in satellite_buffers_algo:
                
                if sat>maximum_data_remaining:
                    max_satellite_index=satellite_counter
                    maximum_data_remaining=sat
                satellite_counter+=1
            #Next need to check if the maximum remaining data is still 0
            if maximum_data_remaining==0:
                actions.append([0,contact[5]])
                #We don't want to send as no satellite that can see the ground station
                #has data to send
            else:
                #Next we will use the satellite that has the most data to send
                actions.append([max_satellite_index+1,contact[5]])
                #next we must update the estimated remaining data
                #Expected Number of packets sent
                satellite_buffers_algo[max_satellite_index]=satellite_buffers_algo[max_satellite_index]-round(contact[0]*contact[1])
                if satellite_buffers_algo[max_satellite_index]<0:
                    satellite_buffers_algo[max_satellite_index]=0
                
                
            
            
        else:
            actions.append([0,contact[5]])
    
    #Next we need to actually implement this algorithm
    #First we need to sort the actions properly
    sorted_actions=sorted(actions,key=lambda row:row[1])
    
    print("Sorted Actions")
    print(sorted_actions)
    #Convert the sorted_actions to a dictionary format
    true_actions={agent: 0 for agent in all_agents}
    index_counter=1
    for agent in all_agents:
        if not (terminated[agent] or truncated[agent]):
            if sorted_actions==index_counter:
                true_actions[agent]=1
    
    
    #Next we actions are actually implemented we check the environment and then conduct sorting and allocation
    #Again
    terminated = {agent: False for agent in all_agents}
    truncated = {agent: False for agent in all_agents}
    
    timestep=0
    while not all([terminated[a] or truncated[a] for a in all_agents]):
        
        index_counter=1
        for agent in all_agents:
            if not (terminated[agent] or truncated[agent]):
                if sorted_actions==index_counter:
                    true_actions[agent]=1
        print("Agent Actions")
        print(true_actions)
        observations, rewards, terminated, truncated, info = env.step(true_actions)
        print(observations)
        #Check if any data was sent
        new_satellite_buffers=observations["satellite1"]
        buffer_change_flag=0
    
        
        #If so we need to recalculate
        if buffer_change_flag==1:
            sat1_observations=init_observations["satellite1"]
            contact_plan=sat1_observations["central_observation_matrix"]
            print(contact_plan)
            #Need to modify the contact plan to only include current observations
            timestep=sat1_observations["current_timestep"]
            updated_contact_plan=[]
            for i in range(timestep,len(contact_plan)):
                updated_contact_plan.append(contact_plan[i])
            
            #Need to add order identifier to each contact
            contact_count=0
            for contact in updated_contact_plan:
                contact.append(contact_count)
                contact_count+=1
            #Need to manage all the satellite buffers and minimize collisions
            #Dictionary that contains the remaining packets to send
            satellite_buffers=sat1_observations["remaining_satellite_data"]
            #Next need to convert this to something easily handled in the heuristic algorithm
            all_agents = ["satellite1", "satellite2", "satellite3"]
            satellite_buffers_algo=[]
            for a in all_agents:
                satellite_buffers_algo.append(satellite_buffers[a])
            
            
            
            #Next we need to sort the contact plan by cloud cover value
            sorted_contact_plan=sorted(updated_contact_plan,key=lambda row: row[0])
            print("Sorted Matrix")
            print(sorted_contact_plan)
            #Next we need to allocate the contacts based upon the buffers of the satellites
            estimated_capacity=0
            for i in sorted_contact_plan:
                if i[0]>= 0:
                    estimated_capacity=round(i[0]*10)
            
            print("Estimated Capacity")
            print(estimated_capacity)
            
            #Now we have our initial action strategy
            #We implement while loop here
            actions=[]#Options are 0,1,2,3
            
            for contact in sorted_contact_plan:
                contact_action_options=[]
                if i[0]>=0:
                    for l in range(0,2):
                        contact_action_options.append(i[2+l])
                    #Next we need to check how much data is left on each satellite that can transmit
                    maximum_data_remaining=0
                    satellite_counter=0
                    max_satellite_index=None
                    for sat in satellite_buffers_algo:
                        
                        if sat>maximum_data_remaining:
                            max_satellite_index=satellite_counter
                            maximum_data_remaining=sat
                        satellite_counter+=1
                    #Next need to check if the maximum remaining data is still 0
                    if maximum_data_remaining==0:
                        actions.append([0,contact[5]])
                        #We don't want to send as no satellite that can see the ground station
                        #has data to send
                    else:
                        #Next we will use the satellite that has the most data to send
                        actions.append([max_satellite_index+1,contact[5]])
                        #next we must update the estimated remaining data
                        #Expected Number of packets sent
                        satellite_buffers_algo[max_satellite_index]=satellite_buffers_algo[max_satellite_index]-round(contact[0]*contact[1])
                        if satellite_buffers_algo[max_satellite_index]<0:
                            satellite_buffers_algo[max_satellite_index]=0
                        
                        
                    
                    
                else:
                    actions.append([0,contact[5]])
            
            #Next we need to actually implement this algorithm
            #First we need to sort the actions properly
            sorted_actions=sorted(actions,key=lambda row:row[1])
            
    
        
        print("Rewards")
        print(rewards)
    Overall_Delivered_Packets=info['Delivered_Packets']
    Overall_Excess_Energy=info['Total_Energy_Expended']
    Overall_Number_of_Contacts_Used=info['Total_Number_of_Contacts_Used']
    agent_specific_delivered_packets=info['satellite_delivered_packets']
    agent_specific_excess_energy=info['satellite_energy_expended']
    agent_specific_num_of_contacts=info['satellite_number_of_contacts_used']
    num_of_collisions=info['Num_of_Collisions']
    env.close()
    return Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts
    env.close()

In [16]:
#CGR
def BaselineCGR(env,seed):
    observations,info = env.reset()
    
    all_agents = ["satellite1", "satellite2", "satellite3"]
    
    terminated = {agent: False for agent in all_agents}
    truncated = {agent: False for agent in all_agents}
    
    while not all([terminated[a] or truncated[a] for a in all_agents]):
        actions = {
            agent:1
            for agent in all_agents
            if not (terminated[agent] or truncated[agent])
        }
        print("Agent Actions")
        print(actions)
        observations, rewards, terminated, truncated, info = env.step(actions)
        print("Rewards")
        print(rewards)
    print(info)
    satellite1_info=info['satellite1']
    Overall_Delivered_Packets=satellite1_info['Delivered_Packets']
    Overall_Excess_Energy=satellite1_info['Total_Energy_Expended']
    Overall_Number_of_Contacts_Used=satellite1_info['Total_Number_of_Contacts_Used']
    num_of_collisions=satellite1_info['Num_of_Collisions']
    i=0
    for a in all_agents:
        sat_info=info[a]
        agent_specific_delivered_packets.append(sat_info['satellite_delivered_packets'])
        agent_specific_excess_energy.append(sat_info['satellite_energy_expended'])
        agent_specific_num_of_contacts.append(sat_info['satellite_number_of_contacts_used'])
        
    env.close()
    return Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts


In [17]:
#Here we do the testing for each algorithm
Number_of_Episodes=100
seeds=[50,51,52,53,54]
CGR_reward_vector=np.zeros(len(seeds)*Number_of_Episodes)
CGR_delivery_ratio_vector=np.zeros(len(seeds)*Number_of_Episodes)
CGR_mean_contact_energy_efficiency_vector=np.zeros(len(seeds)*Number_of_Episodes)
vector_index=0
for h in range(1,Number_of_Episodes):
        for b in seeds:
            total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended=BaselineCGR(env,b)
            vector_index+=1
            CGR_reward_vector[vector_index]=total_reward
            CGR_delivery_ratio_vector[vector_index]=delivered_packets/initial_data_volume
            CGR_mean_contact_energy_efficiency_vector[vector_index]=delivered_packets/number_of_contacts
Rand_reward_vector=np.zeros(len(seeds)*Number_of_Episodes)
Rand_delivery_ratio_vector=np.zeros(len(seeds)*Number_of_Episodes)
Rand_mean_contact_energy_efficiency_vector=np.zeros(len(seeds)*Number_of_Episodes)
vector_index=0
for h in range(1,Number_of_Episodes):
        for b in seeds:
            total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended=RandomPolicy(env,b)
            vector_index+=1
            Rand_reward_vector[vector_index]=total_reward
            Rand_delivery_ratio_vector[vector_index]=delivered_packets/initial_data_volume
            rand_mean_contact_energy_efficiency_vector[vector_index]=delivered_packets/number_of_contacts

Sorting_reward_vector=np.zeros(len(seeds)*Number_of_Episodes)
Sorting_delivery_ratio_vector=np.zeros(len(seeds)*Number_of_Episodes)
Sorting_mean_contact_energy_efficiency_vector=np.zeros(len(seeds)*Number_of_Episodes)
vector_index=0
for h in range(1,Number_of_Episodes):
        for b in seeds:
            total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended=AdaptiveSorting(env,b)
            vector_index+=1
            CGR_reward_vector[vector_index]=total_reward
            CGR_delivery_ratio_vector[vector_index]=delivered_packets/initial_data_volume
            CGR_mean_contact_energy_efficiency_vector[vector_index]=delivered_packets/number_of_contacts





Agent Actions
{'satellite1': 1, 'satellite2': 1, 'satellite3': 1}
Rewards
{'satellite1': -50, 'satellite2': -50, 'satellite3': 0.2857142857142857}
Agent Actions
{'satellite1': 1, 'satellite2': 1, 'satellite3': 1}
Rewards
{'satellite1': 2.2626262626262625, 'satellite2': 2.909090909090909, 'satellite3': 1.6800000000000002}
Agent Actions
{'satellite1': 1, 'satellite2': 1, 'satellite3': 1}
Rewards
{'satellite1': 0.5882352941176471, 'satellite2': 2.653061224489796, 'satellite3': 0.33882352941176475}
Agent Actions
{'satellite1': 1, 'satellite2': 1, 'satellite3': 1}
Rewards
{'satellite1': -50, 'satellite2': -50, 'satellite3': 3.0}
Agent Actions
{'satellite1': 1, 'satellite2': 1, 'satellite3': 1}
Rewards
{'satellite1': -50, 'satellite2': -50, 'satellite3': 3.3600000000000003}
Agent Actions
{'satellite1': 1, 'satellite2': 1, 'satellite3': 1}
Rewards
{'satellite1': 1.7283950617283952, 'satellite2': 2.6984126984126986, 'satellite3': 3.4222222222222216}
Agent Actions
{'satellite1': 1, 'satellite2'

KeyError: 'Total_Number_of_Contacts_Used'

In [ ]:
import matplotlib.pyplot as plt
#Reward Plot
print(np.mean(Sorting_reward_vector))
plt.boxplot([Rand_reward_vector,CGR_reward_vector, Sorting_reward_vector], tick_labels=['Random','CGR', 'Sorting Algorithm'])

plt.title("Rewards ")
plt.ylabel("rewards")
plt.show()

#Delivery Ratio Plot
plt.boxplot([Rand_delivery_ratio_vector,CGR_delivery_ratio_vector, Sorting_delivery_ratio_vector], tick_labels=['Random','CGR', 'Sorting Algorithm'])

plt.title("Delivery Ratio")
plt.ylabel("delivery ratio")
plt.show()
#Contact Energy Efficiency Plot
plt.boxplot([CGR_mean_contact_energy_efficiency_vector, Sorting_mean_contact_energy_efficiency_vector], tick_labels=['CGR', 'Sorting Algorithm'])

plt.title("Box and Whisker Plot")
plt.ylabel("Values")
plt.show()